# Chicago Taxi — Exploration

Run the pipeline first:

```bash
python src/run_pipeline.py --generate
```

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from paths import WAREHOUSE

conn = duckdb.connect(str(WAREHOUSE), read_only=True)
print(f"Connected to {WAREHOUSE}")

In [ ]:
daily = conn.execute("SELECT * FROM chicago_taxi.daily_metrics ORDER BY trip_date").df()
hourly = conn.execute("""
    SELECT hour_of_day, SUM(trip_count) AS trip_count
    FROM chicago_taxi.hourly_demand
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""").df()
fares = conn.execute("SELECT fare FROM chicago_taxi.trips_clean").df()
zones = conn.execute("""
    SELECT pickup_community_area, trip_count, total_revenue
    FROM chicago_taxi.zone_performance
    ORDER BY trip_count DESC
    LIMIT 10
""").df()

daily.head()

## Charts

Replace placeholder takeaways after running on real Chicago data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(daily["trip_date"], daily["trip_count"], marker="o", linewidth=2)
axes[0, 0].set_title("Daily trip volume")
axes[0, 0].set_xlabel("Date")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].bar(hourly["hour_of_day"], hourly["trip_count"], color="steelblue")
axes[0, 1].set_title("Trips by hour of day")
axes[0, 1].set_xlabel("Hour")

axes[1, 0].hist(fares["fare"], bins=30, color="coral", edgecolor="white")
axes[1, 0].set_title("Fare distribution")
axes[1, 0].set_xlabel("Fare ($)")

axes[1, 1].barh(
    zones["pickup_community_area"].astype(str),
    zones["trip_count"],
    color="seagreen",
)
axes[1, 1].set_title("Top 10 pickup zones (by trips)")
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

## Summary (draft)

- **Volume**: [TBD] daily trip pattern across January.
- **Peak hours**: [TBD] highest-demand hours.
- **Economics**: [TBD] typical fare range.
- **Zones**: [TBD] top pickup community areas.

Copy insights into `docs/chicago-taxi-demand-memo.md`.

In [ ]:
conn.close()